---
title: Week 4.6, Group Project: Analaysis of a leachate dataset
subtitle: Landfill Leachate
author:
  - name: Timo Heimovaara
    affiliations: Delft University of Technology, department of Geoscience & Engineering
    orcid: 
    email: t.j.heimovaara@tudelft.nl
license: CC-BY-NC-ND-4.0 (https://creativecommons.org/licenses/by-nc-nd/4.0/).
date: 2026-01-30
kernelspec:
    name: python3
    display_name: 'Python 3.13'
---

## Analsysis of the (partial) chemical composition of landfill leachate

TODO: Introduction
1. Context of the dataset
    - Landfill waste body
    - Methanogenisis in the bulk of the wastebody (PCO2 and PCO2 are about 0.5)
    - Wastebody contains a wide mixture of (reactive) compounds
2. Type of data collected
3. Why this analysis is relevant
4. What type of questions students need to solve
5. Explain what we expect them to do...


In [1]:
# Import libraries required for running all simulations
import os
import sys
from IPython.utils import capture
from IPython.display import display, Markdown
from pathlib import Path
from contextlib import chdir

import numpy as np
import matplotlib.pyplot as plt
import PyORCHESTRA # here, the ORCHESTRA submodule is imported
import pandas as pd
import seaborn as sns

%matplotlib widget
sns.set()

# Prepare a file to capture PyOrchestra output
capture_file = open("pyorchestra_output.log", "w")


# pyOrchestra is implemented in C++
# Save original stdout file descriptor
# original_stdout_fd = sys.stdout.fileno()

# Duplicate original stdout so we can restore it later
# saved_stdout_fd = os.dup(original_stdout_fd)



# We need to import some Orchestra files. We need to know the path layout on 
# the local machine:
def find_book_root(start: Path | None = None) -> Path:
    """
    Walk upward from `start` (or CWD) until a directory containing Jupyter Book
    marker files is found. Returns the path to the book root.
    Raises FileNotFoundError if no root is found.
    """
    config_any = {"_config.yml", "_config.yaml"}      # some projectrs use .yaml
    myst_any = {"myst.yml", "myst.yaml"}            # jupyter-book uses _toc.yml

    cur = Path(Path.cwd()).resolve()

    for parent in [cur, *cur.parents]:
        children = {f.name for f in parent.iterdir()} if parent.exists() else set()
        has_any_config = bool(config_any & children)
        has_any_myst = bool(myst_any & children)
        if has_any_config and has_any_myst:
            return parent

    raise FileNotFoundError(
        f"Could not find Jupyter Book root (no _config.y* and myst.y* found above {cur})"
    )


def path_from_book_root(*parts: str | Path) -> Path:
    root = find_book_root()
    p = (root.joinpath(*parts)).resolve()
    if not p.exists():
        raise FileNotFoundError(f"Path not found: {p}")
    return p

# In order for orchestra to run we need to change directory to the directory with the input file:


# Example usage:
# input_file = path_from_book_root("content", "week 02", "Orchestra_simulation", "chemistry1.inp")
# print("Input file:", input_file)

orchestra_path = path_from_book_root("content", "project_THe", "Orchestra_Project")
# print(orchestra_path)

## Preparation and preliminary investigation of data

TODO: Add a dataset with the leachate production data (cumulative!)

First Week:

1. Import the data
2. Get a quick over view of the content and the structure of the dataset
3. Understand how to plot the time series in the data set, save the figures to a file and create an overview report.
4. Need to calculate molar concentrations from mg/l values
5. Think about what questions you want to resolve with this data?
    - Saturation status of the samples as they are;
    - What were mostly likely conditions where the samples originated?
    - What will happen to the samples if the leachate would be discharged to a system at atmospheric conditions.
5. Prepare the interface to PyOrchestra, have a look at provided GUI of Orchestra and the corresponding Chemistry File


Second Week:
1. Use Orchestra to find out which minerals may be present.
    - please note that Orchestra is an equilibrium calculation


Please note: 
need install openpyxl:  mamba install -c conda-forge openpyxl


We import the data and select the sample with most measurements to do the first Orchestra calculation

In [2]:
# %% 1
# import the data set from the excel file.
# We will use the data from the leachate monitoring at PP-11N.

df_leachate = pd.read_excel('data/df_macros_PP-11N.xlsx')

# %% 2
# Check which dates have most parameters measured.
# We will use these parameters as our input for the Orchestra 
# calculation.

# For some dates, less parameters were measured. We can then choose to work 
# with the estimated values, using the time series.

# df_macro_counts = (
#     df_leachate.group_by('measpointname','date')
#     .agg(pl.count('cname').alias('macro_count'))
#     .sort('macro_count', descending=True)
# )

df_par_counts = (
    df_leachate.groupby(['measpointname', 'date'])['cname']
    .count()
    .reset_index(name='macro_count')
    .sort_values('macro_count', ascending=False)
)

df_par_counts
date_with_most_pars = df_par_counts.iloc[0]['date']

# %%
# We use the date with most parameters for our first analysis
sel_idx = df_leachate['date'] == date_with_most_pars

df_work = df_leachate[sel_idx].copy()

# Export df_work to an Excel file so that we can 
# have a quick access to the parameters in it
# for setting up the translation from mg/l to moles/l 
# for the Orchestra input.
# %%
df_work.to_excel('tmp/df_work_PP-11N.xlsx', index=False)

# %%
# Using the content from the file we now create a table
# with the parameters, their values and the conversion to moles/l.
# We will use this table to set up the translation from mg/l to moles/l
# for the Orchestra input.

# Conversion table contaings molar masses for the parameters 
# in the df_work dataframe and the corresponding parameter names
# in the Orchestra input file.

# componentname, molar mass (g/mol), Orchestra parameter name
conversion_table = {
    'Sulfaat (als SO4)': [96.06, 'SO4-2.tot'],
    'Sulfide': [32.07, 'S-2.tot'],
    'Natrium [Na]': [22.99, 'Na+.tot'],
    'Nikkel [Ni]': [58.69, 'Ni+2.tot'],
    'IJzer [Fe]': [55.85, 'Fe+2.tot'],
    'Zink [Zn]': [65.38, 'Zn+2.tot'],
    'Magnesium [Mg]': [24.31, 'Mg+2.tot'],
    'Calcium [Ca]': [40.08, 'Ca+2.tot'],
    'Ammonium (als NH4)': [18.04, 'NH4+.tot'],
    'Chloride': [35.45, 'Cl-.tot'],
    'Bicarbonaat': [61.02, 'HCO3-.tot'],
    'Fosfaat (als PO4)': [94.97, 'PO4-3.tot'],
    'Kalium [K]': [39.10, 'K+.tot'],
    'Silicium [Si]': [28.09, 'Si.tot'],
    'Mangaan [Mn]': [54.94, 'Mn+2.tot'],
    'Arseen [As]': [74.92, 'As.tot'],
    'Temperatuur': [1e-3, 'T'] # please note the factor 1e-3 which will be corrected for in the conversion to moles/l.
}

# We can now use this conversion table 
# to convert the values in the df_work dataframe
# from mg/l to moles/l and to add a new column with Orchestra parameter names.

df_work['val_mol_l'] = df_work.apply(
    lambda row: 
        (row['val_mgl'] * 1e-3) / conversion_table[row['cname']][0] 
        if row['cname'] in conversion_table else row['val_mgl'], axis=1)

df_work['orchestra_param'] = df_work.apply(
    lambda row: 
        conversion_table[row['cname']][1] 
        if row['cname'] in conversion_table else row['cname'], axis=1)


# select temperatures and add 273.15 to convert to K
sel_temp = df_work['orchestra_param'] == 'T'
df_work.loc[sel_temp, 'val_mol_l'] += 273.15

# %%
# Rewrite df_work to excel so we can copy the contents to the Orchestra input file.
# df_work.to_excel('tmp/df_work_PP-11N_converted.xlsx', index=False)
# %%


## Step 1: Assessing the water samples


In [3]:
df_work

,compartment,measpointname,date,cname,val_mgl,uname_mgl,val_mol_l,orchestra_param
2036,BB11N,PP-11N,2023-12-12,Nikkel [Ni],0.015000,mg/l,2.555802e-07,Ni+2.tot
2037,BB11N,PP-11N,2023-12-12,Chloride,370.000000,mg/l,1.043724e-02,Cl-.tot
2038,BB11N,PP-11N,2023-12-12,Silicium [Si],15.900000,mg/l,5.660377e-04,Si.tot
2039,BB11N,PP-11N,2023-12-12,Temperatuur,18.600000,°C,2.917500e+02,T
2040,BB11N,PP-11N,2023-12-12,Calcium [Ca],410.000000,mg/l,1.022954e-02,Ca+2.tot
2041,BB11N,PP-11N,2023-12-12,Magnesium [Mg],120.000000,mg/l,4.936240e-03,Mg+2.tot
2042,BB11N,PP-11N,2023-12-12,Arseen [As],0.029000,mg/l,3.870796e-07,As.tot
2043,BB11N,PP-11N,2023-12-12,Zink [Zn],0.027000,mg/l,4.129703e-07,Zn+2.tot
2044,BB11N,PP-11N,2023-12-12,Sulfide,0.120000,mg/l,3.741815e-06,S-2.tot
2045,BB11N,PP-11N,2023-12-12,pH,7.160000,-,7.160000e+00,pH


## Initialise the problem 
In order solve this problem with pyOrchestra we first initialize our problem using the *chemistry_Travertine.inp* file. 
This file predefines the aqueous chemical system in such a way that we can use the information from the tables in the paper as inputs to the Orchestra simulation.

Once pyOrchestra is initialized, running a simulation consists of a series of steps where the values of the required set of input variables are passed via *InVARS* to ORCHESTRA, after which a set of corresponding output variables are passed back in *OutVars*.

ORCHESTRA is initialized in pyOrchestra using the inputfile *chemistry_Travertine.inp*, created above with the ORCHESTRA-GUI. After initialization in Python, we know which variables will be passed through *OutVars* and can be used in *InVars*.

The following code implements these steps.


In [4]:
# We get the list of primary states from df_work and the corresponding parameter names in 
# the Orchestra input file.

primary_states = df_work['orchestra_param'].tolist()
print(primary_states)

# We can now copy the content of this list to the InVars1 list for the Orchestra interface.

# We obtain the required outputs for the dissolved species using the OrchestraGUI

out_logact_diss = [
    'Ar.logact', 'AsO4-3.logact', 'CO2.logact', 'CO3-2.logact', 'Ca+2.logact', 
    'CaCO3.logact', 'CaHCO3+.logact', 'CaOH+.logact', 'CaSO4.logact', 
    'Ca[HPO4].logact', 'Ca[OH]+.logact', 'Ca[SO4].logact', 'Cl-.logact', 
    'Fe+2.logact', 'FeCO3.logact', 'FeCl+.logact', 'FeCl2.logact', 'FeCl3-.logact', 
    'Fe[CO3]2-2.logact', 'Fe[H2PO4]+.logact', 'Fe[HPO4].logact', 'Fe[HS]+.logact', 
    'Fe[HS]2.logact', 'Fe[NH3]+2.logact', 'Fe[NH3]2+2.logact', 'Fe[NH3]4+2.logact', 
    'Fe[OH]+.logact', 'Fe[OH]2.logact', 'Fe[OH]3-.logact', 'Fe[OH]4-2.logact', 
    'Fe[SO4].logact', 'H+.logact', 'H2CO3.logact', 'H2S.logact', 'H2[AsO4]-.logact', 
    'H2[PO4]-.logact', 'H2[SiO4]-2.logact', 'H3[AsO4].logact', 'H3[PO4].logact', 
    'H3[SiO4]-.logact', 'H4[SiO4].logact', 'HCO3-.logact', 'HPO4-2.logact', 
    'HS-.logact', 'HSO4-.logact', 'H[AsO4]-2.logact', 'K+.logact', 'KPO4-2.logact', 
    'KSO4-.logact', 'K[HPO4]-.logact', 
    'Mg+2.logact', 'MgCO3.logact', 'MgHCO3+.logact', 'MgOH+.logact', 'MgSO4.logact', 
    'Mg[H2PO4]+.logact', 'Mg[H3SiO4]+.logact', 'Mg[HPO4].logact', 'Mg[NH3]+2.logact', 
    'Mg[NH3]2+2.logact', 'Mg[NH3]3+2.logact', 'Mg[NH3]4+2.logact', 'Mg[PO4]-.logact', 
    'Mn+2.logact', 'Mn2[OH]+3.logact', 'Mn2[OH]3+.logact', 'MnCl+.logact', 'MnCl2.logact', 
    'MnCl3-.logact', 'Mn[CO3].logact', 'Mn[HCO3]+.logact', 'Mn[HPO4].logact', 
    'Mn[HPO4]2-2.logact', 'Mn[NH3]+2.logact', 'Mn[NH3]2+2.logact', 'Mn[OH]+.logact', 
    'Mn[OH]2.logact', 'Mn[OH]3-.logact', 'Mn[OH]4-2.logact', 'Mn[SO4].logact', 
    'NH3.logact', 'NH4+.logact', 'Na+.logact', 'NaCO3-.logact', 'NaH2PO4.logact', 
    'NaHCO3.logact', 'NaPO4-2.logact', 'NaSO4-.logact', 'Na[HPO4]-.logact', 
    'Ni+2.logact', 'Ni2[OH]+3.logact', 'Ni4[OH]4+4.logact', 'NiCl+.logact', 
    'NiHAsO4.logact', 'NiHS+.logact', 'Ni[CO3].logact', 'Ni[CO3]2-2.logact', 
    'Ni[HCO3]+.logact', 'Ni[HPO4].logact', 'Ni[HS]2.logact', 'Ni[NH3]+2.logact', 
    'Ni[NH3]2+2.logact', 'Ni[NH3]3+2.logact', 'Ni[NH3]4+2.logact', 'Ni[OH]+.logact', 
    'Ni[OH]2.logact', 'Ni[OH]2[HPO4]-2.logact', 'Ni[SO4].logact', 'Ni[SO4]2-2.logact', 
    'OH-.logact', 'PO4-3.logact', 'S-2.logact', 'SO4-2.logact', 'Si2O2[OH]5-.logact', 
    'Si2O3[OH]4-2.logact', 'Si3O5[OH]5-3.logact', 'Si3O6[OH]3-3.logact', 
    'Si4O6[OH]6-2.logact', 'Si4O7[OH]6-4.logact', 'Si4O8[OH]4-4.logact', 
    'Si6O15-6.logact', 'Zn+2.logact'
]

out_con_diss = [
    'Ar.con', 'AsO4-3.con', 'CO2.con', 'CO3-2.con', 'Ca+2.con', 
    'CaCO3.con', 'CaHCO3+.con', 'CaOH+.con', 'CaSO4.con', 
    'Ca[HPO4].con', 'Ca[OH]+.con', 'Ca[SO4].con', 'Cl-.con', 
    'Fe+2.con', 'FeCO3.con', 'FeCl+.con', 'FeCl2.con', 'FeCl3-.con', 
    'Fe[CO3]2-2.con', 'Fe[H2PO4]+.con', 'Fe[HPO4].con', 'Fe[HS]+.con', 
    'Fe[HS]2.con', 'Fe[NH3]+2.con', 'Fe[NH3]2+2.con', 'Fe[NH3]4+2.con', 
    'Fe[OH]+.con', 'Fe[OH]2.con', 'Fe[OH]3-.con', 'Fe[OH]4-2.con', 
    'Fe[SO4].con', 'H+.con', 'H2CO3.con', 'H2S.con', 'H2[AsO4]-.con', 
    'H2[PO4]-.con', 'H2[SiO4]-2.con', 'H3[AsO4].con', 'H3[PO4].con', 
    'H3[SiO4]-.con', 'H4[SiO4].con', 'HCO3-.con', 'HPO4-2.con', 
    'HS-.con', 'HSO4-.con', 'H[AsO4]-2.con', 'K+.con', 'KPO4-2.con', 
    'KSO4-.con', 'K[HPO4]-.con', 
    'Mg+2.con', 'MgCO3.con', 'MgHCO3+.con', 'MgOH+.con', 'MgSO4.con', 
    'Mg[H2PO4]+.con', 'Mg[H3SiO4]+.con', 'Mg[HPO4].con', 'Mg[NH3]+2.con', 
    'Mg[NH3]2+2.con', 'Mg[NH3]3+2.con', 'Mg[NH3]4+2.con', 'Mg[PO4]-.con', 
    'Mn+2.con', 'Mn2[OH]+3.con', 'Mn2[OH]3+.con', 'MnCl+.con', 'MnCl2.con', 
    'MnCl3-.con', 'Mn[CO3].con', 'Mn[HCO3]+.con', 'Mn[HPO4].con', 
    'Mn[HPO4]2-2.con', 'Mn[NH3]+2.con', 'Mn[NH3]2+2.con', 'Mn[OH]+.con', 
    'Mn[OH]2.con', 'Mn[OH]3-.con', 'Mn[OH]4-2.con', 'Mn[SO4].con', 
    'NH3.con', 'NH4+.con', 'Na+.con', 'NaCO3-.con', 'NaH2PO4.con', 
    'NaHCO3.con', 'NaPO4-2.con', 'NaSO4-.con', 'Na[HPO4]-.con', 
    'Ni+2.con', 'Ni2[OH]+3.con', 'Ni4[OH]4+4.con', 'NiCl+.con', 
    'NiHAsO4.con', 'NiHS+.con', 'Ni[CO3].con', 'Ni[CO3]2-2.con', 
    'Ni[HCO3]+.con', 'Ni[HPO4].con', 'Ni[HS]2.con', 'Ni[NH3]+2.con', 
    'Ni[NH3]2+2.con', 'Ni[NH3]3+2.con', 'Ni[NH3]4+2.con', 'Ni[OH]+.con', 
    'Ni[OH]2.con', 'Ni[OH]2[HPO4]-2.con', 'Ni[SO4].con', 'Ni[SO4]2-2.con', 
    'OH-.con', 'PO4-3.con', 'S-2.con', 'SO4-2.con', 'Si2O2[OH]5-.con', 
    'Si2O3[OH]4-2.con', 'Si3O5[OH]5-3.con', 'Si3O6[OH]3-3.con', 
    'Si4O6[OH]6-2.con', 'Si4O7[OH]6-4.con', 'Si4O8[OH]4-4.con', 
    'Si6O15-6.con', 'Zn+2.con'   
]

out_si_minerals = [
    'Anhydrite[s].si', 'Aragonite[s].si', 'Calcite[s].si', 'Dolomite[s].si', 'FeS[ppt][s].si', 
    'Gypsum[s].si', 'Halite[s].si', 'Hydroxyapatite[s].si', 'Mackinawite[s].si', 
    'Melanterite[s].si', 'Pyrochroite[s].si', 'Quartz[s].si', 'Rhodochrosite[s].si', 
    'Siderite[s].si', 'Smithsonite[s].si', 'Sphalerite[s].si', 'Sylvite[s].si', 
    'Talc[s].si', 'Vivianite[s].si', 'Zn[OH]2[e][s].si'
]

out_tot_minerals = [
    'Anhydrite[s].tot', 'Aragonite[s].tot', 'Calcite[s].tot', 'Dolomite[s].tot', 'FeS[ppt][s].tot', 
    'Gypsum[s].tot', 'Halite[s].tot', 'Hydroxyapatite[s].tot', 'Mackinawite[s].tot', 
    'Melanterite[s].tot', 'Pyrochroite[s].tot', 'Quartz[s].tot', 'Rhodochrosite[s].tot', 
    'Siderite[s].tot', 'Smithsonite[s].tot', 'Sphalerite[s].tot', 'Sylvite[s].tot', 
    'Talc[s].tot', 'Vivianite[s].tot', 'Zn[OH]2[e][s].tot'
]

out_logact_gases = ['Ar[g].logact', 'CO2[g].logact', 'Ar[g].tot', 'CO2[g].tot']

out_extra = ['chargebalance', 'I', 'totcharge']

InVars_no_precip = [
    'gas_type', 'gasvolume_fixed', 
    'Ar[g].logact',
    'As.tot', 'Ca+2.tot', 'Cl-.tot', 'Fe+2.tot', 'HCO3-.tot',
    'K+.tot', 'Mg+2.tot', 'Mn+2.tot', 'NH4+.tot', 'Na+.tot',
    'Ni+2.tot', 'PO4-3.tot', 'S-2.tot', 'SO4-2.tot', 'Si.tot',
    'Zn+2.tot',
    'T', 'pH', 'watervolume'
]

# For the simulations with precipitation we need to add fixed_logact_CO2 to control the CO2[g].logactivity.
# Please note we need add sufficient CO2 to the system to allow the fixed_logact_CO2 to reach equilibrium,
# this is done by overuling the HCO3-.tot value. Please note Orchestra compensate the pH using the chargebalance.

InVars_precip = [
    'fixed_logact_CO2', 'gas_type', 'gasvolume_fixed', 
    'Ar[g].logact',
    'As.tot', 'Ca+2.tot', 'Cl-.tot', 'Fe+2.tot', 'HCO3-.tot',
    'K+.tot', 'Mg+2.tot', 'Mn+2.tot', 'NH4+.tot', 'Na+.tot',
    'Ni+2.tot', 'PO4-3.tot', 'S-2.tot', 'SO4-2.tot', 'Si.tot',
    'Zn+2.tot',
    'T', 'pH', 'watervolume'
]

['Ni+2.tot', 'Cl-.tot', 'Si.tot', 'T', 'Ca+2.tot', 'Mg+2.tot', 'As.tot', 'Zn+2.tot', 'S-2.tot', 'pH', 'K+.tot', 'NH4+.tot', 'Mn+2.tot', 'HCO3-.tot', 'Na+.tot', 'PO4-3.tot', 'SO4-2.tot', 'Fe+2.tot']


In [5]:
#--- Initialize problem --- 
# for initialization we need to temporarily move to the directory containing the 'chemistry1.inp' file.
# Later we use pO1, InVars1 and OutVars1
with chdir(orchestra_path):

    # Input file is generated with Orchestra GUI
    InputFile = 'chemistry_leachate_no_precip.inp'
    NoCells = 1 #only 1 cell to have a 0-D system with 1liter of water
    
    # We define the input variables that will be changed in the script
    # We use the same sequence as used in the paper
    InVars1 = np.array(InVars_no_precip)
    
    # We select the output from Orchestra we need to use
    # We use the Output selector tab in the GUI to select the output variables.
    out_list = (
        primary_states + out_logact_diss + out_si_minerals + 
        out_tot_minerals + out_logact_gases + out_con_diss + 
        out_extra
    )
    OutVars1 = np.array(out_list)
        

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO1 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO1.initialise(InputFile, NoCells, InVars1, OutVars1)

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO1 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO1.initialise(InputFile, NoCells, InVars1, OutVars1)


Reading and expanding calculator new stylechemistry_leachate_no_precip.inp
Scanning file: chemistry_leachate_no_precip.inp
Scanning file: objects2025_THe.txt
Including file: objects2025_THe.txt
Scanning file: chemistry_leachate_no_precip.inp
Scanning file: objects2025_THe.txt
Including file: objects2025_THe.txt
Including file: chemistry_leachate_no_precip.inp
Scanning file: objects2025_THe.txt
Including file: objects2025_THe.txt
0.096 sec.
	Reading variables .... 0.067 s
testing:
41:gas_type
42:gasvolume_fixed
6:Ar[g].logact
8:As.tot
10:Ca+2.tot
12:Cl-.tot
14:Fe+2.tot
17:HCO3-.tot
19:K+.tot
21:Mg+2.tot
23:Mn+2.tot
25:NH4+.tot
27:Na+.tot
29:Ni+2.tot
31:PO4-3.tot
33:S-2.tot
35:SO4-2.tot
37:Si.tot
39:Zn+2.tot
4:T
15:pH
43:watervolume
29:Ni+2.tot
12:Cl-.tot
37:Si.tot
4:T
10:Ca+2.tot
21:Mg+2.tot
8:As.tot
39:Zn+2.tot
33:S-2.tot
15:pH
19:K+.tot
25:NH4+.tot
23:Mn+2.tot
17:HCO3-.tot
27:Na+.tot
31:PO4-3.tot
35:SO4-2.tot
14:Fe+2.tot
44:Ar.logact
45:AsO4-3.logact
46:CO2.logact
47:CO3-2.logact
9:Ca

Please note that the output of this code is what ORCHESTRA echos back. ORCHESTRA uses a set of variables in order to store the input variables and the results of the calculations, in this case 33. The top part of the output shows the output requested by us through *OutVars* together with the values used during initialization.

### Run the Problem

1. We use the data imported as the mass of our primary states.
2. We check the output for relevant species. 
    - we exported all possible species. We can negelect all species with very low activities;
    - we exported all possible minerals. We only need to include the ones with relatively high SI-values > -0.5?



In [6]:
# Let us print the initial primary state values from the database

table_md_data = df_work.sort_values(['orchestra_param'])[['orchestra_param','val_mol_l']].to_markdown()
display(Markdown(table_md_data))

# print(df_data)
#df_work.sort_values(['orchestra_param'])['orchestra_param'].to_list()

138:Ni[OH]+.logact
139:Ni[OH]2.logact
140:Ni[OH]2[HPO4]-2.logact
141:Ni[SO4].logact
142:Ni[SO4]2-2.logact
143:OH-.logact
30:PO4-3.logact
32:S-2.logact
34:SO4-2.logact
144:Si2O2[OH]5-.logact
145:Si2O3[OH]4-2.logact
146:Si3O5[OH]5-3.logact
147:Si3O6[OH]3-3.logact
148:Si4O6[OH]6-2.logact
149:Si4O7[OH]6-4.logact
150:Si4O8[OH]4-4.logact
151:Si6O15-6.logact
38:Zn+2.logact
152:Anhydrite[s].si
153:Aragonite[s].si
154:Calcite[s].si
155:Dolomite[s].si
156:FeS[ppt][s].si
157:Gypsum[s].si
158:Halite[s].si
159:Hydroxyapatite[s].si
160:Mackinawite[s].si
161:Melanterite[s].si
162:Pyrochroite[s].si
163:Quartz[s].si
164:Rhodochrosite[s].si
165:Siderite[s].si
166:Smithsonite[s].si
167:Sphalerite[s].si
168:Sylvite[s].si
169:Talc[s].si
170:Vivianite[s].si
171:Zn[OH]2[e][s].si
172:Anhydrite[s].tot
173:Aragonite[s].tot
174:Calcite[s].tot
175:Dolomite[s].tot
176:FeS[ppt][s].tot
177:Gypsum[s].tot
178:Halite[s].tot
179:Hydroxyapatite[s].tot
180:Mackinawite[s].tot
181:Melanterite[s].tot
182:Pyrochroite[s].tot
1

|      | orchestra_param   |     val_mol_l |
|-----:|:------------------|--------------:|
| 2042 | As.tot            |   3.8708e-07  |
| 2040 | Ca+2.tot          |   0.0102295   |
| 2037 | Cl-.tot           |   0.0104372   |
| 2053 | Fe+2.tot          |   7.52014e-05 |
| 2049 | HCO3-.tot         |   0.042609    |
| 2046 | K+.tot            |   0.00511509  |
| 2041 | Mg+2.tot          |   0.00493624  |
| 2048 | Mn+2.tot          |   1.82017e-05 |
| 2047 | NH4+.tot          |   0.0206385   |
| 2050 | Na+.tot           |   0.0173989   |
| 2036 | Ni+2.tot          |   2.5558e-07  |
| 2051 | PO4-3.tot         |   7.88945e-05 |
| 2044 | S-2.tot           |   3.74181e-06 |
| 2052 | SO4-2.tot         |   0.00801193  |
| 2038 | Si.tot            |   0.000566038 |
| 2039 | T                 | 291.75        |
| 2043 | Zn+2.tot          |   4.1297e-07  |
| 2045 | pH                |   7.16        |

In [7]:
# %%
# Run the model using chemistry_Travertine.inp for the above samples
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
# prepare output matrix
all_Res = np.zeros([1,len(OutVars1)])

for param in df_work['orchestra_param']:
    # print(param)
    IN1[0][np.where(InVars1 == param)[0][0]] = df_work.loc[df_work['orchestra_param'] == param, 'val_mol_l'].values[0]

# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'fixed_logact_CO2')] = 0 # fixed CO2 logact
IN1[0][np.where(InVars1 == 'Ar[g].logact')] = -20 #   # no gasvolume no background gas 
IN1[0][np.where(InVars1 == 'gas_type')] = 0 # fixed_gasvolume
IN1[0][np.where(InVars1 == 'gasvolume_fixed')] = 1e-20 # 1 atm pressure

# run ORCHESTRA
OUT = pO1.set_and_calculate(IN1)
all_Res = OUT[0]

# Create a dataframe from all_Res
Res_Simulation = pd.DataFrame([all_Res],columns=OutVars1, index=['first calculation'])

table_mdini = Res_Simulation[[
    'pH', 'chargebalance','totcharge','Ar[g].logact', 'CO2[g].logact','HCO3-.tot', 'HCO3-.logact', 
    'Ca+2.logact', 'Calcite[s].si', 'Gypsum[s].si',
    'HCO3-.con', 'CO3-2.con']].to_markdown()
display(Markdown(table_mdini))


|                   |   pH |   chargebalance |   totcharge |   Ar[g].logact |   CO2[g].logact |   HCO3-.tot |   HCO3-.logact |   Ca+2.logact |   Calcite[s].si |   Gypsum[s].si |   HCO3-.con |   CO3-2.con |
|:------------------|-----:|----------------:|------------:|---------------:|----------------:|------------:|---------------:|--------------:|----------------:|---------------:|------------:|------------:|
| first calculation | 7.16 |       0.0128444 |   0.0839994 |            -20 |        -0.97371 |    0.042609 |       -1.59119 |      -2.56923 |         1.06372 |      -0.648273 |   0.0317434 |  3.7253e-05 |

## Steps we need to analyse this data

### Aims of the analysis:
1. What is the current state of the sample. Which minerals are supersaturated? How does the partial pressure of CO2 compare with the CO2 concentration in the atmosphere and what may be expected of the CO2 pressure in the waste body.
2. What will be the changes when the sample is moved to a situation where it is completely in equilibrium with the atmosphere?
3. What is a likely composition of the leachate from which this sample originates? Which minerals control the composition? We may assume a partial CO2 pressure of 0.5 atm

## Aim 1: current state of the sample

From the output above we see that the pH is different from the one reported by the laboratory (pH laboratory = 7.16), that the CO2[g].logact is -2.27 which indicates a higher partial pressure of CO2[g] than in the atmosphere (logP = -3.37) and that the SI-value for Calcite is positive. All these indicate that the laboratory analysis was from a sample that is not in equilibrium.

There are some first checks that need to be done:
1. Check the charge balance in the sample in order to assess the quality of the sampling and analysis
   

In [8]:
# Calculate the Electrical Balance in % from the results
EB = Res_Simulation['chargebalance']/Res_Simulation['totcharge'] * 100

print(f"The chargebalance is {Res_Simulation['chargebalance'].values[0]} and the total charge in the system is {Res_Simulation['totcharge'].values[0]}. ")
print(f"The electrical balance is therefore {EB.values[0]} %. ")

The chargebalance is 0.012844403274357319 and the total charge in the system is 0.08399941772222519. 
The electrical balance is therefore 15.291062355041504 %. 


The electrical balance is about 8%, this is not perfect and is also an indication of that some errors occured because of sampling and or analysis. When a sample is taken, it is split in to to fractions:
- for the cations, to which a strong acid is added to prevent precipitation of solids;
- for the anions. Here we cannot add acid, as this will lead to degassing of CO2.

Given the type of sample and the first assessment, we decide to accept the sample.

2. The next analysis is to list the minerals with SI-values that are larger than -0.5. These minerals may have been present in the wastebody where the sample originated.

In [9]:
# List all minerals with SI-values > 0.5

sel_large_SI = Res_Simulation.loc['first calculation', out_si_minerals] > -0.5

selected_minerals = sel_large_SI[sel_large_SI].index.tolist()
display(Markdown(Res_Simulation[selected_minerals].to_markdown()))


|                   |   Aragonite[s].si |   Calcite[s].si |   Dolomite[s].si |   FeS[ppt][s].si |   Hydroxyapatite[s].si |   Mackinawite[s].si |   Quartz[s].si |   Rhodochrosite[s].si |   Siderite[s].si |   Sphalerite[s].si |   Vivianite[s].si |
|:------------------|------------------:|----------------:|-----------------:|-----------------:|-----------------------:|--------------------:|---------------:|----------------------:|-----------------:|-------------------:|------------------:|
| first calculation |          0.868329 |         1.06372 |          1.93404 |         0.350215 |                7.66167 |           -0.374785 |       0.573263 |             -0.162329 |         0.905803 |            6.32257 |           1.07775 |

Clearly quite a number of these minerals are superstaturated. What we can do is to run a simulation where we allow the above listed minerals to precipitate.
This requires a modification of the chemistry input file of Orchestra. We have applied this modification to the chemistry_leachate_precip.inp file.

After modification of the chemistry_leachate_precip.inp file, we initialize a second orchestra instance for this input file and rerun the simulation.
We save the output to a new record in the Res_Simulation dataframe.


In [10]:
sel_large_SI

Anhydrite[s].si         False
Aragonite[s].si          True
Calcite[s].si            True
Dolomite[s].si           True
FeS[ppt][s].si           True
Gypsum[s].si            False
Halite[s].si            False
Hydroxyapatite[s].si     True
Mackinawite[s].si        True
Melanterite[s].si       False
Pyrochroite[s].si       False
Quartz[s].si             True
Rhodochrosite[s].si      True
Siderite[s].si           True
Smithsonite[s].si       False
Sphalerite[s].si         True
Sylvite[s].si           False
Talc[s].si              False
Vivianite[s].si          True
Zn[OH]2[e][s].si        False
Name: first calculation, dtype: bool

In [11]:
#--- Initialize problem --- 
# for initialization we need to temporarily move to the directory containing the 'chemistry1.inp' file.
# Later we use pO1, InVars1 and OutVars1
with chdir(orchestra_path):

    # Input file is generated with Orchestra GUI
    InputFile = 'chemistry_leachate_precip.inp'
    NoCells = 1 #only 1 cell to have a 0-D system with 1liter of water
    
    # We define the input variables that will be changed in the script
    # We use the same sequence as used in the paper
    InVars1 = np.array(InVars_precip)
    
    # We select the output from Orchestra we need to use
    # We use the Output selector tab in the GUI to select the output variables.
    
    out_list = (
        primary_states + out_logact_diss + out_si_minerals + 
        out_tot_minerals + out_logact_gases + out_con_diss + 
        out_extra
    )
    OutVars1 = np.array(out_list)
        

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO2 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO2.initialise(InputFile, NoCells, InVars1, OutVars1)

    # Associate a variable with the pyOrchestra.ORCHESTRA() class
    pO2 = PyORCHESTRA.ORCHESTRA()

    # Initialize the class with the parameters defined above
    pO2.initialise(InputFile, NoCells, InVars1, OutVars1)


Try a first calculation with iia switched off....
Parsing expressions of chemistry_leachate_no_precip.inp..... 
Optimizing expressions of chemistry_leachate_no_precip.inp..... 0.349 sec.
8372 variables, 24881 expressions, 17 equations.
First calculation was successful!
Repeat calculation with iia switched on..
Switching on: logI: -2
This was successful!!
Reading and expanding calculator new stylechemistry_leachate_precip.inp
Scanning file: chemistry_leachate_precip.inp
Scanning file: objects2025_THe.txt
Including file: objects2025_THe.txt
Scanning file: chemistry_leachate_precip.inp
Scanning file: objects2025_THe.txt
Including file: objects2025_THe.txt
Including file: chemistry_leachate_precip.inp
Scanning file: objects2025_THe.txt
Including file: objects2025_THe.txt
0.12 sec.
	Reading variables .... 0.073 s
testing:
54:fixed_logact_CO2
55:gas_type
56:gasvolume_fixed
7:Ar[g].logact
9:As.tot
11:Ca+2.tot
13:Cl-.tot
15:Fe+2.tot
18:HCO3-.tot
20:K+.tot
22:Mg+2.tot
24:Mn+2.tot
26:NH4+.tot
28

In [12]:
# %%
# Run the model using chemistry_Travertine.inp for the above samples
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
# prepare output matrix
all_Res = np.zeros([1,len(OutVars1)])

for param in df_work['orchestra_param']:
    # print(param)
    IN1[0][np.where(InVars1 == param)[0][0]] = df_work.loc[df_work['orchestra_param'] == param, 'val_mol_l'].values[0]

# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'fixed_logact_CO2')] = -3.37 # fixed CO2 logact atmospheric CO2 pressure
IN1[0][np.where(InVars1 == 'Ar[g].logact')] = -20 # background pressure = 1 atm 
IN1[0][np.where(InVars1 == 'gas_type')] = 0 # fixed_gasvolume
IN1[0][np.where(InVars1 == 'gasvolume_fixed')] = 1e-20 # 1 atm pressure

# We need to make sure that sufficient CO2 is present in the system to allow the system to reach the specified CO2[g].logact
# We add CO2 with CO3-2.tot
IN1[0][np.where(InVars1 == 'HCO3-.tot')] = 10 # Excess of CO2, will end up as precipitates 
# and in the virtual CO2g[s] mineral which serves as a store for CO2[g]


# run ORCHESTRA
OUT = pO2.set_and_calculate(IN1)
all_Res = OUT[0]

# Create a dataframe from all_Res
Res_Simulation.loc['second calculation'] = all_Res



logact : -3
13 : Cl-.tot : 1e-09
14 : Fe+2.logact : -9
15 : Fe+2.tot : 0.01
16 : pH : 7
17 : HCO3-.logact : -9
18 : HCO3-.tot : 0.01
19 : K+.logact : -9
20 : K+.tot : 1e-09
21 : Mg+2.logact : -9
22 : Mg+2.tot : 1e-09
23 : Mn+2.logact : -9
24 : Mn+2.tot : 0.01
25 : NH4+.logact : -9
26 : NH4+.tot : 0.01
27 : Na+.logact : -9
28 : Na+.tot : 1e-09
29 : Ni+2.logact : -9
30 : Ni+2.tot : 0.01
31 : PO4-3.logact : -9
32 : PO4-3.tot : 0.01
33 : S-2.logact : -9
34 : S-2.tot : 0.01
35 : SO4-2.logact : -9
36 : SO4-2.tot : 1e-09
37 : Si.logact : -9
38 : Si.tot : 0.01
39 : Zn+2.logact : -9
40 : Zn+2.tot : 0.01
41 : gas_val_u : 0.1
42 : Aragonite[s].un : -0.001
43 : CO2g[s].un : -0.001
44 : Calcite[s].un : -0.001
45 : Dolomite[s].un : -0.001
46 : FeS[ppt][s].un : -0.001
47 : Hydroxyapatite[s].un : -0.001
48 : Mackinawite[s].un : -0.001
49 : Quartz[s].un : -0.001
50 : Rhodochrosite[s].un : -0.001
51 : Siderite[s].un : -0.001
52 : Sphalerite[s].un : -0.001
53 : Vivianite[s].un : -0.001
54 : fixed_logact_

Clearly the concentrations have changed by allowing the precipitation.
We can repeat the analysis for the SI-values to see which minerals have precipitated and which are subsaturated.


In [13]:
table_mdini = Res_Simulation[[
    'T','pH', 'chargebalance','totcharge','Ar[g].logact', 'CO2[g].logact','HCO3-.tot', 'HCO3-.logact', 
    'Ca+2.logact', 'Calcite[s].si', 'Gypsum[s].si',
    'HCO3-.con', 'CO3-2.con']].to_markdown()
display(Markdown(table_mdini))

|                    |      T |      pH |   chargebalance |   totcharge |   Ar[g].logact |   CO2[g].logact |   HCO3-.tot |   HCO3-.logact |   Ca+2.logact |   Calcite[s].si |   Gypsum[s].si |   HCO3-.con |   CO3-2.con |
|:-------------------|-------:|--------:|----------------:|------------:|---------------:|----------------:|------------:|---------------:|--------------:|----------------:|---------------:|------------:|------------:|
| first calculation  | 291.75 | 7.16    |       0.0128444 |   0.0839994 |            -20 |        -0.97371 |    0.042609 |       -1.59119 |      -2.56923 |     1.06372     |      -0.648273 |  0.0317434  | 3.7253e-05  |
| second calculation | 291.75 | 9.05541 |       0         |   0.0645749 |            -20 |        -3.37    |   10        |       -2.09279 |      -5.01943 |    -8.88178e-16 |      -2.85539  |  0.00967271 | 0.000802393 |

In [14]:
# List all minerals with SI-values > 0.5

sel_large_SI = Res_Simulation.loc['second calculation', out_si_minerals] >=-0.5

selected_minerals = sel_large_SI[sel_large_SI].index.tolist()
display(Markdown(Res_Simulation[selected_minerals].to_markdown()))

# print(sel_large_SI)


|                    |   Aragonite[s].si |   Calcite[s].si |   Dolomite[s].si |   FeS[ppt][s].si |   Hydroxyapatite[s].si |   Quartz[s].si |   Rhodochrosite[s].si |   Siderite[s].si |   Sphalerite[s].si |
|:-------------------|------------------:|----------------:|-----------------:|-----------------:|-----------------------:|---------------:|----------------------:|-----------------:|-------------------:|
| first calculation  |          0.868329 |     1.06372     |      1.93404     |        0.350215  |                7.66167 |       0.573263 |             -0.162329 |      0.905803    |        6.32257     |
| second calculation |         -0.148707 |    -8.88178e-16 |     -1.77636e-15 |       -0.0240526 |                0       |       0        |             -0.145016 |     -8.88178e-16 |        1.77636e-15 |

### Estimation of the leachate composition within the waste body
Using the above simulation results we can now use the model to make an educated guess of the leachate composition within the wastebody.

We make two major assumptions for this calculation:
1. The CO2[g] pressure within the waste body is much higher than in the atmosphere. If we assume anaerobic conditions where methanogenic conditions occur we may estimate the CO[g] pressure to be 0.5 atm with a total pressure of about 1 atm.
2. The minerals which have relatively large SI-values will defnitely be present in the waste body.

We now need to adjust the master variables in such a way that we can have the model simulate the conditions in the waste body.

The compositions of the following minerals can be found from the input file of Orchestra and are:
- Calcite[s]: CaCO3
- FeS[ppt][s]: FeS
- Hydroxyapatite[s]: Ca5(PO4)3(OH)
- Quartz[s]: SiO2
- Rhodochrosite[s]: MnCO3
- Siderite[s]: FeCO3
- Sphalerite[s]: ZnS

We assume that above minerals are present at 10 moles per liter.
So we change the input values of the master-species as follows:

Ca+2.tot == 10 + 50 = 60
Fe+2.tot == 20 
S-2.tot == 20
Si.tot == 10
Mn+2.tot == 10
Zn+2.tot == 10
PO4-3.tot == 30
HCO3-.tot == 30

The CO2 pressure is equal to 0.5 atm, so fixed_log_act == -0.301


In [15]:
# %%
# Run the model using chemistry_Travertine.inp for the above samples
IN1 = np.array([np.ones_like(InVars1)]).astype(float)
# prepare output matrix
all_Res = np.zeros([1,len(OutVars1)])

for param in df_work['orchestra_param']:
    # print(param)
    IN1[0][np.where(InVars1 == param)[0][0]] = df_work.loc[df_work['orchestra_param'] == param, 'val_mol_l'].values[0]

# set default watervolume and gasvolume
IN1[0][np.where(InVars1 == 'watervolume')] = 1.0 # per liter
IN1[0][np.where(InVars1 == 'fixed_logact_CO2')] = -0.301 # fixed CO2 logact atmospheric CO2 pressure
IN1[0][np.where(InVars1 == 'Ar[g].logact')] = -20 # background pressure = 1 atm 
IN1[0][np.where(InVars1 == 'gas_type')] = 0 # fixed_gasvolume
IN1[0][np.where(InVars1 == 'gasvolume_fixed')] = 1e-20 # 1 atm pressure

# We need to make sure that sufficient CO2 is present in the system to allow the system to reach the specified CO2[g].logact
# We add CO2 with CO3-2.tot

IN1[0][np.where(InVars1 == 'Ca+2.tot')] = 60
IN1[0][np.where(InVars1 == 'Fe+2.tot')] = 20 
IN1[0][np.where(InVars1 == 'S-2.tot')] = 20
IN1[0][np.where(InVars1 == 'Si.tot')] = 10
IN1[0][np.where(InVars1 == 'Mn+2.tot')] = 10
IN1[0][np.where(InVars1 == 'Zn+2.tot')] = 10
IN1[0][np.where(InVars1 == 'PO4-3.tot')] = 30
IN1[0][np.where(InVars1 == 'HCO3-.tot')] = 30 # Excess of CO2, will end up as precipitates 
# and in the virtual CO2g[s] mineral which serves as a store for CO2[g]


# run ORCHESTRA
OUT = pO2.set_and_calculate(IN1)
all_Res = OUT[0]

# Create a dataframe from all_Res
Res_Simulation.loc['third calculation'] = all_Res


In [16]:
table_mdini = Res_Simulation[[
    'T','pH', 'chargebalance','totcharge','Ar[g].logact', 'CO2[g].logact','HCO3-.tot', 'HCO3-.logact', 
    'Ca+2.logact', 'Calcite[s].si', 'Gypsum[s].si',
    'HCO3-.con', 'CO3-2.con']].to_markdown()
display(Markdown(table_mdini))

|                    |      T |       pH |   chargebalance |   totcharge |   Ar[g].logact |   CO2[g].logact |   HCO3-.tot |   HCO3-.logact |   Ca+2.logact |   Calcite[s].si |   Gypsum[s].si |   HCO3-.con |   CO3-2.con |
|:-------------------|-------:|---------:|----------------:|------------:|---------------:|----------------:|------------:|---------------:|--------------:|----------------:|---------------:|------------:|------------:|
| first calculation  | 291.75 |  7.16    |       0.0128444 |   0.0839994 |            -20 |        -0.97371 |    0.042609 |       -1.59119 |      -2.56923 |     1.06372     |      -0.648273 | 0.0317434   | 3.7253e-05  |
| second calculation | 291.75 |  9.05541 |       0         |   0.0645749 |            -20 |        -3.37    |   10        |       -2.09279 |      -5.01943 |    -8.88178e-16 |      -2.85539  | 0.00967271  | 0.000802393 |
| third calculation  | 291.75 | 11.134   |       0         |   0.118845  |            -20 |        -9.31387 |   30        |       -5.95805 |      -3.23277 |    -4.44089e-16 |      -1.107    | 1.30117e-06 | 1.23798e-05 |

In [17]:
sel_large_SI = Res_Simulation.loc['second calculation', out_si_minerals] >=-0.5

selected_minerals = sel_large_SI[sel_large_SI].index.tolist()
display(Markdown(Res_Simulation[selected_minerals].to_markdown()))

print(sel_large_SI)


|                    |   Aragonite[s].si |   Calcite[s].si |   Dolomite[s].si |   FeS[ppt][s].si |   Hydroxyapatite[s].si |   Quartz[s].si |   Rhodochrosite[s].si |   Siderite[s].si |   Sphalerite[s].si |
|:-------------------|------------------:|----------------:|-----------------:|-----------------:|-----------------------:|---------------:|----------------------:|-----------------:|-------------------:|
| first calculation  |          0.868329 |     1.06372     |      1.93404     |        0.350215  |            7.66167     |       0.573263 |             -0.162329 |      0.905803    |        6.32257     |
| second calculation |         -0.148707 |    -8.88178e-16 |     -1.77636e-15 |       -0.0240526 |            0           |       0        |             -0.145016 |     -8.88178e-16 |        1.77636e-15 |
| third calculation  |         -0.148707 |    -4.44089e-16 |     -8.88178e-16 |        0         |            1.42109e-14 |       0        |              0        |      0           |        0           |

Anhydrite[s].si         False
Aragonite[s].si          True
Calcite[s].si            True
Dolomite[s].si           True
FeS[ppt][s].si           True
Gypsum[s].si            False
Halite[s].si            False
Hydroxyapatite[s].si     True
Mackinawite[s].si       False
Melanterite[s].si       False
Pyrochroite[s].si       False
Quartz[s].si             True
Rhodochrosite[s].si      True
Siderite[s].si           True
Smithsonite[s].si       False
Sphalerite[s].si         True
Sylvite[s].si           False
Talc[s].si              False
Vivianite[s].si         False
Zn[OH]2[e][s].si        False
Name: second calculation, dtype: bool


In [19]:
table_mdini = Res_Simulation[[
    'HCO3-.con','CO3-2.con', 'H2CO3.con', 
    'PO4-3.con', 'S-2.con',
    ]].to_markdown()
display(Markdown(table_mdini))

|                    |   HCO3-.con |   CO3-2.con |   H2CO3.con |   PO4-3.con |     S-2.con |
|:-------------------|------------:|------------:|------------:|------------:|------------:|
| first calculation  | 0.0317434   | 3.7253e-05  | 0.00432507  | 5.04825e-10 | 2.23533e-16 |
| second calculation | 0.00967271  | 0.000802393 | 4.93393e-39 | 2.93333e-09 | 2.71546e-14 |
| third calculation  | 1.30117e-06 | 1.23798e-05 | 5.60519e-45 | 5.50102e-13 | 4.42814e-16 |

## Optimal output with totals of the master_species present in the solution
I need to check if I can use the .solution for the master species as input for my final calculations where I want to estimate the mass of precipitates forming from this leachate.